In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import mode
from util.DataGen import nai_pulse
import scipy.signal as signal

dir = 'Data'

In [ ]:
def read_trace_file(path):
    with open(path, 'r') as file:
        file_content = file.readlines()
    data_line = 4
    chunks = eval(file_content[data_line][2:])
    freeze, tracesample = chunks
    trace = np.array(chunks[tracesample])
    
    # the ADC is 12bits so 2^12 = 4096 = 1000mV.  The trace data you have is divided by 16 i.e. 4096/16 = 256. 
    # So you need to multiply the trace data by 16 to convert it back to full bit precision
    # and  if you want the data in mV you multiply it by 1000/4096 .
    # TODO consider if we should convert to volts...
    
    return trace

In [ ]:
print(len(os.listdir(dir)))
print(os.listdir(dir))

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(10,2), dpi=200)

for scintillator, axis in zip(['nai', 'pla'], axes):
    mins = []
    maxes = []
    modes = []
    for file in os.listdir(dir):
        if not ".xtr" in file or scintillator not in file:
            continue
            
        trace = read_trace_file(os.path.join(dir, file))
        mins.append(np.min(trace))
        maxes.append(np.max(trace))
        modes.append(mode(trace, keepdims=False)[0])
        
    axis.plot(mins, '*', label=scintillator + ' mins', alpha=.5)
    axis.plot(maxes, '*', label=scintillator + ' maxes', alpha=.5)
    axis.plot(modes, '.r', markersize=3, label=scintillator + ' mode', alpha=1)

    axis.legend()
plt.show()
        

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5), dpi=200)

for scintillator, axis in zip(['nai', 'pla'], axes):
    for file in os.listdir(dir):
        if not ".xtr" in file or scintillator not in file:
            continue

        trace = read_trace_file(os.path.join(dir, file))
        axis.plot(trace)

    axis.set_title(scintillator)
    axis.legend()
plt.show()


In [ ]:
_, old_kernel_estimate = nai_pulse(1)
kernel_len = old_kernel_estimate.size
dist_to_peak = np.argmax(old_kernel_estimate)

files = os.listdir(dir)

wide = 0
total = 0

fig, axes = plt.subplots(1, 2, figsize=(10, 3), dpi=200)
for file in files:
    if not ".xtr" in file or 'nai' not in file:
        continue
        
    trace = read_trace_file(os.path.join(dir, file))
    peaks = signal.find_peaks(trace)[0]
    
    trace_mode = mode(trace, keepdims=False)[0]
    # print(trace_mode)
    
    peaks = peaks[trace[peaks] - trace_mode > 5]
    for peak in peaks:
        event = trace[peak-dist_to_peak: peak-dist_to_peak+kernel_len]
        zero_baseline = event - trace_mode
        if (
            np.sum(event==0) > 3
            or np.sum(event==255) > 1
            or np.argmax(event) != dist_to_peak
            or zero_baseline[0] >= 1
            or np.sum(zero_baseline==np.max(zero_baseline)) > 1
            or np.max(zero_baseline) <= 10
        ):
            continue
        axes[0].plot(zero_baseline)
        
        normed = zero_baseline / np.max(zero_baseline)
        total += 1
        if np.sum(normed == 1) > 1:
            wide += 1
        axes[1].plot(normed, alpha=.3)

axes[0].set_title('NaI Responses')
axes[1].set_title('NaI Normed Responses')
plt.show()

print('Portion of peaks that are wide:', wide / total)
# note this may be reduced by selecting for larger peaks which are better for characterization anyway..

In [ ]:
# Could we model this as a weighted average of each index in the kernels, where the weighting somehow takes into account the relative magntiude
# of the individual kernel peak to that bin width?
# This data is better for averaging than THOR because of the lower probability of low energy gammas. Is that true?

# New idea: average signal as the inverse to account for time shifts! aka: given each V magnitude along curve (Y), average T values (X) to find the optimal time. This does not account for discretization
# This also assumed uniform distribution in time, which is a good assumption with more samples than we have. Might be nice to weight cruve constrivution to average based on the ratio of its magnitude to bit width. This is to try to account for discretization.

# Spline Interpolation? Interpolation filter?
# make_smoothing_spline

# Expectaion Maximization! Single Cluster in upsampled space. Alternate mean estimation discrete((shift, mag)) estimation
# Loop:
#     shifted = shift(reponses) # init zero shift
#     mean_est = mean(normed(shifted))
#     mean_error = round(discretize())
#     ... IDK



In [ ]:
_, old_kernel_estimate = nai_pulse(1)
kernel_len = old_kernel_estimate.size
dist_to_peak = np.argmax(old_kernel_estimate)

files = os.listdir(dir)

wide = 0
total = 0

fig, axes = plt.subplots(1, 2, figsize=(10, 3), dpi=200)
for file in files:
    if not ".xtr" in file or 'pla' not in file:
        continue
        
    trace = read_trace_file(os.path.join(dir, file))
    peaks = signal.find_peaks(trace)[0]
    
    trace_mode = mode(trace, keepdims=False)[0]
    # print(trace_mode)
    
    peaks = peaks[trace[peaks] - trace_mode > 5]
    for peak in peaks:
        event = trace[peak-dist_to_peak: peak-dist_to_peak+kernel_len]
        zero_baseline = event - trace_mode
        if (
            np.sum(event==0) > 3
            or np.sum(event==255) > 1
            or np.argmax(event) != dist_to_peak
            or zero_baseline[0] >= 1
            or np.sum(zero_baseline==np.max(zero_baseline)) > 1
            or not np.any(zero_baseline[np.argmax(zero_baseline):np.argmax(zero_baseline)+10] < 0)
            or np.max(zero_baseline) <= 6
        ):
            continue
        axes[0].plot(zero_baseline)
        
        normed = zero_baseline / np.max(zero_baseline)
        total += 1
        if np.sum(normed == 1) > 1:
            wide += 1
        axes[1].plot(normed, alpha=.5)

axes[0].set_title('Pla Responses')
axes[1].set_title('Pla Normed Responses')
axes[1].set_xlim(0, 25)
plt.show()

print('Portion of peaks that are wide:', wide / total)
# note this may be reduced by selecting for larger peaks which are better for characterization anyway..


In [ ]:
# Explore muon shutdown
_, old_kernel_estimate = nai_pulse(1)
kernel_len = old_kernel_estimate.size
dist_to_peak = np.argmax(old_kernel_estimate)

files = os.listdir(dir)

count = 0

fig, axes = plt.subplots(2, 1, figsize=(10, 7), dpi=200)
for file in files:
    if not ".xtr" in file or 'nai' not in file:
        continue
        
    trace = read_trace_file(os.path.join(dir, file))
    mn = np.min(trace)
    if (
        np.sum(trace == 255) <= 1
    ):
        continue
    
    max_where = np.where(trace == 255)[0]
    # print(max_where)
    # print(np.diff(max_where))
    if max_where.size == 0:
        continue
    
    mx_diff = np.where(np.diff(max_where) > 1)[0]  
    sat_starts = max_where[mx_diff+1] # need the right hand side of the diff function
    sat_starts = np.concatenate((np.array([max_where[0]]), sat_starts)).astype(int) # include first one
    # print(seg_starts)
    
    for sat_start in sat_starts:
        sat_end = sat_start + 1
        # print(seg_start, seg_end)
        
        while sat_end < trace.size and trace[sat_end] == 255:
            sat_end += 1
        seg_end = sat_end
        # sat_end -= 1
        while seg_end < trace.size and trace[seg_end] != 0:
            seg_end += 1
        while seg_end < trace.size and trace[seg_end] == 0:
            seg_end += 1
        if np.any(trace[seg_end:seg_end+100] == 255):
            continue # avoid plotting oscillations getting cliped multiple times at highs
        if seg_end - sat_start > 50:
            seg_end = sat_start + 50
    
        # print(sat_start, sat_end, seg_end)
        # print(sat_start-sat_start, sat_end-sat_start, seg_end-sat_start)
        
        n = 1
        index = np.arange(trace.size)[sat_start-n : seg_end+3]
        event = trace[index]
        
        count+=1
        
        # if count != 25:
        #     continue
        
        axes[0].plot(event, alpha=.2)
        axes[0].plot(n, trace[sat_start], 'g*')
        axes[0].plot(sat_end-sat_start, trace[sat_end-1], 'r*')
        
        n = 5
        index = np.arange(trace.size)[sat_end-n : sat_end+40]
        event = trace[index]
        axes[1].plot(event, alpha=.2)
        axes[1].plot(n-1, trace[sat_end-1], 'r*')
        
        
    #     if count > 1:
    #         break
    # if count > 1:
    #     break 
 

axes[0].set_title('NaI Saturated Responses aligned by Saturation Start')
axes[1].set_title('NaI Saturated Responses aligned by Saturation End')
plt.show()

In [ ]:
 # TODO Below
"""
 Question: What is the cause of different clusters of behavior

 Gather traces first before plotting
 Calculate sat_start, sat_end, seg_end and store
 
 try clustering the above based on points after saturation end to constant.
 Plot traces together by cluster. Is there commonality in the pre-sat_end saturation behavior?
 
 Can we try to reverse engineer the components in the amplifier based on these signals?
 
"""

In [ ]:
# file = os.listdir(dir)[-7]
# print(file)
# trace = read_trace_file(os.path.join(dir, file))
# 
# plt.figure(figsize=(10,4), dpi=200)
# plt.plot(trace, 'b.')
# # plt.yscale('log')
# plt.xlim(3900, 4200)
# # plt.xlim(23000, 25000)
# # plt.ylim(0, 300)